In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_pickle(r"C:\Users\Almog\Desktop\Data Science\Projects\2026\Customer Personality Analysis\Pickle files\FS_MC1.pkl")
df.head()

,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,tenure_years,Education_ord,Marital_Status_encoded
0,1957,58138.0,0,0,58,635,88,546,172,88,...,0,0,0,0,0,0,1,1.815,2,2
1,1954,46344.0,1,1,38,11,1,6,2,1,...,0,0,0,0,0,0,0,0.309,2,2
2,1965,71613.0,0,0,26,426,49,127,111,21,...,0,0,0,0,0,0,0,0.854,2,1
3,1984,26646.0,1,0,26,11,4,20,10,3,...,0,0,0,0,0,0,0,0.381,2,1
4,1981,58293.0,1,0,94,173,43,118,46,27,...,0,0,0,0,0,0,0,0.441,4,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Year_Birth              2240 non-null   int64  
 1   Income                  2240 non-null   float64
 2   Kidhome                 2240 non-null   int64  
 3   Teenhome                2240 non-null   int64  
 4   Recency                 2240 non-null   int64  
 5   MntWines                2240 non-null   int64  
 6   MntFruits               2240 non-null   int64  
 7   MntMeatProducts         2240 non-null   int64  
 8   MntFishProducts         2240 non-null   int64  
 9   MntSweetProducts        2240 non-null   int64  
 10  MntGoldProds            2240 non-null   int64  
 11  NumDealsPurchases       2240 non-null   int64  
 12  NumWebPurchases         2240 non-null   int64  
 13  NumCatalogPurchases     2240 non-null   int64  
 14  NumStorePurchases       2240 non-null   

#### Feature Transformation
#### Detect Which Features Need Transformation

In [8]:
from scipy.stats import skew

# Numeric columns only
num_cols = df.select_dtypes(include=['int64', 'float64', 'int32', 'Int64']).columns

results = []

for col in num_cols:
    data = df[col].dropna()

    sk = skew(data)

    results.append({
        'feature': col,
        'skewness': round(sk, 2),
        'needs_transformation': abs(sk) > 1
    })

skewness_df = pd.DataFrame(results).sort_values(
    by='skewness',
    ascending=False
)

skewness_df

,feature,skewness,needs_transformation
21,Complain,10.18,True
20,AcceptedCmp2,8.47,True
1,Income,6.80,True
19,AcceptedCmp1,3.55,True
16,AcceptedCmp3,3.29,True
18,AcceptedCmp5,3.29,True
17,AcceptedCmp4,3.24,True
11,NumDealsPurchases,2.42,True
9,MntSweetProducts,2.13,True
6,MntFruits,2.10,True


**Feature Transformation - Skewness Analysis**

To prepare the dataset for clustering, we evaluated the skewness of all numeric features in order to identify variables with highly asymmetric distributions.

The analysis shows that several features are strongly right-skewed, particularly campaign-response variables (`AcceptedCmp1-5`, `Response`, `Complain`) and monetary spending features (`Income`, `MntWines`, `MntMeatProducts`, `MntSweetProducts`, etc.). These variables contain long tails and extreme values that may distort distance calculations during clustering.

Features with absolute skewness greater than 1 were marked as candidates for transformation. In particular:
- `Complain`, `AcceptedCmp2`, and `Income` exhibit extremely high skewness.
- Spending-related variables and purchase-count variables also show substantial positive skewness.
- Features such as `Recency`, `tenure_years`, `Year_Birth`, and household-related variables are relatively well-behaved and likely do not require transformation.

Based on these findings, the next step will involve applying appropriate transformations (primarily logarithmic transformation) to heavily skewed variables before scaling and clustering.

#### Apply Transformations

In [14]:
# Features selected for log transformation
log_features = [
    'Income',
    'MntWines',
    'MntFruits',
    'MntMeatProducts',
    'MntFishProducts',
    'MntSweetProducts',
    'MntGoldProds',
    'NumDealsPurchases',
    'NumWebPurchases',
    'NumCatalogPurchases'
]

# Apply log1p transformation
for col in log_features:
    df[f'{col}_log'] = np.log1p(df[col])

# Preview transformed columns
df[[f'{col}_log' for col in log_features]].head()

,Income_log,MntWines_log,MntFruits_log,MntMeatProducts_log,MntFishProducts_log,MntSweetProducts_log,MntGoldProds_log,NumDealsPurchases_log,NumWebPurchases_log,NumCatalogPurchases_log
0,10.970592,6.455199,4.488636,6.304449,5.153292,4.488636,4.488636,1.386294,2.197225,2.397895
1,10.743869,2.484907,0.693147,1.945910,1.098612,0.693147,1.945910,1.098612,0.693147,0.693147
2,11.179046,6.056784,3.912023,4.852030,4.718499,3.091042,3.761200,0.693147,2.197225,1.098612
3,10.190432,2.484907,1.609438,3.044522,2.397895,1.386294,1.791759,1.098612,1.098612,0.000000
4,10.973254,5.159055,3.784190,4.779123,3.850148,3.332205,2.772589,1.791759,1.791759,1.386294


**Feature Transformation - Logarithmic Scaling**

Following the skewness analysis, logarithmic transformation was applied to heavily right-skewed monetary and behavioral features using the `log1p()` function.

The purpose of this transformation is to:
- Compress extreme values and long-tailed distributions
- Reduce the dominance of large observations in distance calculations
- Improve the geometric balance of the clustering space

The `log1p()` transformation was selected because it safely handles zero values while preserving the relative ordering of observations.

New transformed columns were created instead of overwriting the original variables in order to preserve interpretability and allow later comparison between raw and transformed distributions.

The transformed features primarily include:
- Income-related variables
- Spending features
- Purchase-count activity features

These transformed variables will later be evaluated and scaled prior to clustering.

#### Compare Before vs After Transformation

In [18]:
comparison_results = []

for col in log_features:
    
    original_skew = skew(df[col].dropna())
    transformed_skew = skew(df[f'{col}_log'].dropna())
    
    comparison_results.append({
        'feature': col,
        'original_skewness': round(original_skew, 2),
        'log_skewness': round(transformed_skew, 2),
        'improvement': round(original_skew - transformed_skew, 2)
    })

transformation_comparison_df = pd.DataFrame(comparison_results)

transformation_comparison_df.sort_values(
    by='improvement',
    ascending=False
)

,feature,original_skewness,log_skewness,improvement
0,Income,6.80,-1.18,7.98
6,MntGoldProds,1.88,-0.34,2.23
3,MntMeatProducts,2.08,-0.08,2.16
5,MntSweetProducts,2.13,0.09,2.05
2,MntFruits,2.10,0.08,2.02
4,MntFishProducts,1.92,-0.05,1.97
7,NumDealsPurchases,2.42,0.67,1.75
9,NumCatalogPurchases,1.88,0.13,1.75
1,MntWines,1.17,-0.55,1.72
8,NumWebPurchases,1.38,-0.26,1.65


**Feature Transformation Evaluation**

To evaluate the effectiveness of the logarithmic transformation, skewness was measured before and after applying the `log1p()` transformation to the selected features.

The results show a substantial reduction in skewness across all transformed variables, indicating that the distributions became significantly more balanced and geometrically suitable for clustering.

Key observations:
- `Income` showed the largest improvement, with skewness decreasing from `6.80` to `-1.18`.
- Spending-related variables (`MntMeatProducts`, `MntSweetProducts`, `MntFruits`, `MntFishProducts`, `MntGoldProds`) were transformed from heavily skewed distributions into approximately symmetric distributions.
- Purchase activity variables (`NumDealsPurchases`, `NumCatalogPurchases`, `NumWebPurchases`) also showed meaningful reduction in skewness.

Overall, the transformation step successfully compressed long tails and reduced the dominance of extreme values, improving the geometric balance of the feature space prior to scaling and clustering.

#### Keeping the transformed features and dropping the old ones

In [22]:
# Original features to remove
original_features_to_drop = [
    'Income',
    'MntWines',
    'MntFruits',
    'MntMeatProducts',
    'MntFishProducts',
    'MntSweetProducts',
    'MntGoldProds',
    'NumDealsPurchases',
    'NumCatalogPurchases',
    'NumWebPurchases'
]

# Drop original versions
df = df.drop(columns=original_features_to_drop)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Year_Birth               2240 non-null   int64  
 1   Kidhome                  2240 non-null   int64  
 2   Teenhome                 2240 non-null   int64  
 3   Recency                  2240 non-null   int64  
 4   NumStorePurchases        2240 non-null   int64  
 5   NumWebVisitsMonth        2240 non-null   int64  
 6   AcceptedCmp3             2240 non-null   int64  
 7   AcceptedCmp4             2240 non-null   int64  
 8   AcceptedCmp5             2240 non-null   int64  
 9   AcceptedCmp1             2240 non-null   int64  
 10  AcceptedCmp2             2240 non-null   int64  
 11  Complain                 2240 non-null   int64  
 12  Response                 2240 non-null   int64  
 13  tenure_years             2240 non-null   float64
 14  Education_ord           

#### Scaling

In [25]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Scale the full dataset
df_scaled = scaler.fit_transform(df)

# Convert back to DataFrame
df_scaled = pd.DataFrame(df_scaled, columns=df.columns)

# Preview
df_scaled.head()

,Year_Birth,Kidhome,Teenhome,Recency,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,...,Income_log,MntWines_log,MntFruits_log,MntMeatProducts_log,MntFishProducts_log,MntSweetProducts_log,MntGoldProds_log,NumDealsPurchases_log,NumWebPurchases_log,NumCatalogPurchases_log
0,-0.985345,-0.825218,-0.929894,0.307039,-0.550785,0.693904,-0.28014,-0.28383,-0.28014,-0.262111,...,0.428874,0.987816,1.432838,1.395245,1.580292,1.411082,1.061778,0.653324,1.281507,1.805279
1,-1.235733,1.032559,0.906934,-0.383664,-1.166125,-0.130463,-0.28014,-0.28383,-0.28014,-0.262111,...,-0.021774,-1.213337,-0.984548,-1.397218,-0.866776,-0.970316,-0.912490,0.042760,-1.396187,-0.405464
2,-0.317643,-0.825218,-0.929894,-0.798086,1.295237,-0.542647,-0.28014,-0.28383,-0.28014,-0.262111,...,0.843209,0.766933,1.065587,0.464698,1.317887,0.534192,0.496970,-0.817780,1.281507,0.120349
3,1.268149,1.032559,-0.929894,-0.798086,-0.550785,0.281720,-0.28014,-0.28383,-0.28014,-0.262111,...,-1.121817,-1.213337,-0.400953,-0.693351,-0.082637,-0.535416,-1.032178,0.042760,-0.674342,-1.304348
4,1.017761,1.032559,-0.929894,1.550305,0.064556,-0.130463,-0.28014,-0.28383,-0.28014,-0.262111,...,0.434166,0.269227,0.984169,0.417988,0.793822,0.685504,-0.270625,1.513864,0.559662,0.493420


**Feature Scaling**

After completing feature transformation and final feature selection, all variables were standardized using `StandardScaler`.

The purpose of scaling is to ensure that all features contribute fairly to distance calculations during clustering. Without scaling, variables with larger numerical ranges could dominate the geometry of the feature space and bias the clustering results.

`StandardScaler` transforms each feature so that:
- The mean becomes approximately `0`
- The standard deviation becomes approximately `1`

This process places all variables on a comparable scale while preserving the relative differences between observations.

At this stage, the dataset is fully prepared for clustering:
- Missing values were handled
- Features were encoded
- New behavioral features were engineered
- Noisy variables were removed
- Skewed distributions were transformed
- All features were scaled into a common geometric space

In [32]:
df_scaled.to_pickle(r"C:\Users\Almog\Desktop\Data Science\Projects\2026\Customer Personality Analysis\Pickle files\FTS_MC1.pkl")